# HACKOWEEK SEM 5 - Week 7 & Week 8
## Supervised Machine Learning: Regression & Classification

**Student:** Samruddhi Kalbande  
**Course:** B.Tech Computer Science / Information Technology (5th Semester)  
**Topics Covered:**
1. **Regression Algorithms:**
   - Linear Regression (Ordinary Least Squares)
   - Polynomial Regression (Degree 2 with interactions)
   - Ridge Regression ($L_2$ Regularization)
   - Lasso Regression ($L_1$ Regularization & Feature Selection)
2. **Classification Algorithms:**
   - Logistic Regression (Probabilistic Decision Boundary)
   - K-Nearest Neighbors (KNN Classification with Distance Metrics)
3. **Dataset:** Curated Kaggle Student Performance Dataset (`kaggle_student_performance.csv`)

### 1. Data Ingestion & Feature Preprocessing
Loading student records and scaling features (`StandardScaler`) to prevent gradient distortion in Ridge and Lasso.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)

# Load Kaggle student performance dataset
df = pd.read_csv('../data/kaggle_student_performance.csv')

features = ['StudyHoursPerWeek', 'AttendanceRate', 'MathScore', 'ReadingScore', 'WritingScore']
X = df[features]
y_reg = df['CGPA']
y_clf = (df['CGPA'] >= 8.5).astype(int)  # 1 = High Performer, 0 = Standard

# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Loaded {len(df)} student profiles.")
print(f"Class balance (High Performers): {np.sum(y_clf)}/{len(y_clf)}")
df[['StudentID', 'Name'] + features + ['CGPA']].head(3)

---
## 2. Regression Part A: Linear & Polynomial Regression
- **Linear Regression:** Fits hyper-plane minimizing $\sum (y_i - \mathbf{w}^T \mathbf{x}_i - b)^2$.
- **Polynomial Regression:** Expands feature space with nonlinear powers and interactions $\mathbf{x}_i \mathbf{x}_j$.

In [2]:
# 1. Linear Regression
lr = LinearRegression()
lr.fit(X_scaled, y_reg)
y_pred_lr = lr.predict(X_scaled)

# 2. Polynomial Regression (Degree 2)
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_scaled)
poly_reg = LinearRegression()
poly_reg.fit(X_poly, y_reg)
y_pred_poly = poly_reg.predict(X_poly)

print("=== Linear Regression ===")
print(f"R² Score: {r2_score(y_reg, y_pred_lr):.4f} | MSE: {mean_squared_error(y_reg, y_pred_lr):.4f}")
for f, w in zip(features, lr.coef_):
    print(f"  Weight [{f}]: {w:+.4f}")

print("\n=== Polynomial Regression (Degree 2) ===")
print(f"R² Score: {r2_score(y_reg, y_pred_poly):.4f} | MSE: {mean_squared_error(y_reg, y_pred_poly):.4f}")
print(f"Expanded Feature Count: {X_poly.shape[1]} terms")

---
## 3. Regression Part B: Regularized Regression (Ridge vs. Lasso)
- **Ridge ($L_2$ Penalty):** Minimizes $\text{MSE} + \alpha \sum w_j^2$. Shrinks weights smoothly toward zero to prevent multicollinearity.
- **Lasso ($L_1$ Penalty):** Minimizes $\text{MSE} + \alpha \sum |w_j|$. Drives non-essential weights strictly to zero, performing automatic feature selection.

In [3]:
# Train Ridge and Lasso
ridge = Ridge(alpha=1.0).fit(X_scaled, y_reg)
lasso = Lasso(alpha=0.05, max_iter=2000).fit(X_scaled, y_reg)

y_pred_ridge = ridge.predict(X_scaled)
y_pred_lasso = lasso.predict(X_scaled)

coef_comparison = pd.DataFrame({
    'Feature': features,
    'Linear (OLS)': lr.coef_,
    'Ridge (L2)': ridge.coef_,
    'Lasso (L1)': lasso.coef_
}).round(4)

print("Coefficient Comparison (Notice shrinkage in Ridge, Sparsity in Lasso):")
print(coef_comparison)

# Bar chart comparison
coef_comparison.set_index('Feature').plot(kind='bar', figsize=(9, 4.5))
plt.title('Regression Coefficients Comparison across Regularization Types')
plt.ylabel('Weight Magnitude')
plt.xticks(rotation=20)
plt.show()

---
## 4. Classification: Logistic Regression & K-Nearest Neighbors
Predicting whether a student belongs to the High Performer / Distinction category ($y \in \{0, 1\}$).
- **Logistic Regression:** Outputs probability $P(y=1|\mathbf{x}) = \frac{1}{1 + e^{-(\mathbf{w}^T \mathbf{x} + b)}}$.
- **K-Nearest Neighbors (KNN):** Non-parametric algorithm assigning majority vote among the $k$ closest neighbors in Euclidean feature space.

In [4]:
# 1. Logistic Regression
clf_log = LogisticRegression().fit(X_scaled, y_clf)
y_pred_log = clf_log.predict(X_scaled)
y_prob_log = clf_log.predict_proba(X_scaled)[:, 1]

# 2. KNN Classifier (k=3)
clf_knn = KNeighborsClassifier(n_neighbors=3).fit(X_scaled, y_clf)
y_pred_knn = clf_knn.predict(X_scaled)

print("=== Logistic Regression Performance ===")
print(f"Accuracy: {accuracy_score(y_clf, y_pred_log)*100:.1f}%")
print(classification_report(y_clf, y_pred_log, target_names=['Standard', 'High Performer']))

print("=== K-Nearest Neighbors (k=3) Performance ===")
print(f"Accuracy: {accuracy_score(y_clf, y_pred_knn)*100:.1f}%")
print(classification_report(y_clf, y_pred_knn, target_names=['Standard', 'High Performer']))

In [5]:
# Confusion Matrix Visualization
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

cm_log = confusion_matrix(y_clf, y_pred_log)
sns.heatmap(cm_log, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Standard', 'High'], yticklabels=['Standard', 'High'])
axes[0].set_title('Logistic Regression Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

cm_knn = confusion_matrix(y_clf, y_pred_knn)
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Purples', ax=axes[1],
            xticklabels=['Standard', 'High'], yticklabels=['Standard', 'High'])
axes[1].set_title('KNN (k=3) Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

---
## 5. Live Prediction Demonstration
Applying the trained models to forecast academic standing and CGPA for an unseen candidate student.

In [6]:
# Candidate: 19 hrs study/wk, 93% attendance, Math: 92, Reading: 89, Writing: 91
sample_raw = np.array([[19.0, 93.0, 92, 89, 91]])
sample_scaled = scaler.transform(sample_raw)

cgpa_pred_lr = lr.predict(sample_scaled)[0]
cgpa_pred_ridge = ridge.predict(sample_scaled)[0]
cgpa_pred_lasso = lasso.predict(sample_scaled)[0]
prob_high = clf_log.predict_proba(sample_scaled)[0][1]

print("--- Candidate Student Prediction ---")
print(f"Linear Regression CGPA Forecast: {cgpa_pred_lr:.2f}")
print(f"Ridge Regression CGPA Forecast:  {cgpa_pred_ridge:.2f}")
print(f"Lasso Regression CGPA Forecast:  {cgpa_pred_lasso:.2f}")
print(f"Probability of High Performer (Distinction): {prob_high*100:.1f}%")
print(f"Final Classification: {'Distinction Candidate' if prob_high >= 0.5 else 'Standard Standing'}")